# nb51 - Containment-fraction auxiliary head (Act 1b, H14)

**Error analysis.** nb50: widening the window does NOT reduce the residual-containment correlation (0.57/0.60/0.55 at W=4/6/8) - the containment term is longitudinal leakage / sampling loss that never reaches any window. Collecting more cells is dead; the leakage must be PREDICTED from shower shape and corrected.

**Question.** Does supervising the encoder to predict the true contained fraction (label available on clean events only) improve the minbias energy resolution through the shared representation?

**Hypothesis.** H14: adding an auxiliary head that regresses c = sumE_window/(1000*Etrue) on the clean-auxiliary rows of the nb44 winning stack teaches the encoder shower-shape -> leakage, improving sigma_eff at E>17 GeV.

**Research.** CRILIN software compensation: shape observables strongly correlate with the contained fraction and event-by-event corrections from them substantially improve resolution (arXiv:2606.05111); CALICE learned per-cell compensation (arXiv:2403.04632). Multi-task caveat from our own record: H7 showed pressure on the GATE hurts - this head shares only the ENCODER, leaving gate and energy head free.

**Proof criterion.** Config identical to nb44 quantaux except the aux head (lambda_c = 0.5 fixed a priori, aux loss on clean rows only); 2 seeds; anchors = nb44 singles 0.0442 +/- 0.0005 (seeds 0-4), ens 0.0425 stack record. Win = beat singles mean by >0.002 overall or in any E>17 bin. Diagnostic: corr(chat, c) on clean val rows (does the head learn containment) and corr(residual, containment proxy) on minbias test (does the correction transfer).

In [1]:
import os, sys, copy, time, pathlib
import numpy as np, pandas as pd
import torch, torch.nn as nn
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import split, resolution, PITCH, EPS
from picocal_data import build_grid, prep
OUT = REPO / 'reports' / 'predictions'
CKPT = REPO / '.scratch' / 'ckpt'; CKPT.mkdir(parents=True, exist_ok=True)
DEVICE = os.environ.get('NB51_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB51_MODE', 'full')
MBF = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
CLF = sorted((REPO / 'data' / 'full').glob('matched_*.root'))
if MODE == 'smoke': MBF, CLF = MBF[:8], CLF[:4]
t0 = time.time()
ME = build_grid(MBF, 'minbias')
CE = build_grid(CLF, 'clean')
D = prep(4, ME, CE, ng=6)
sumE_all = np.expm1(np.array([0.0]))
print(f'device {DEVICE} | mode {MODE} | build+prep {time.time()-t0:.0f}s')

minbias: 72554 events


clean: 30303 events


W=4: N 102857 (main 72554 + aux 30303), tr/va/te 50787/10883/10884, IN_DIM 16
device cuda | mode full | build+prep 145s


In [2]:
NG = 6
is_clean = D['G'][:, 5] > 0
sumW = D['Eraw'].sum(1)
CONT = np.clip(sumW / (1000.0 * D['Et']), 0.0, 2.0).astype(np.float32)
T = dict(X=torch.from_numpy(D['X']).to(DEVICE), M=torch.from_numpy(D['M']).to(DEVICE),
         G=torch.from_numpy(D['G']).to(DEVICE), Y=torch.from_numpy(D['y']).unsqueeze(1).to(DEVICE),
         E=torch.from_numpy(D['Eraw']).to(DEVICE), C=torch.from_numpy(CONT).to(DEVICE),
         IC=torch.from_numpy(is_clean.astype(np.float32)).to(DEVICE))
ktr, kva, kte, ctr = D['ktr'], D['kva'], D['kte'], D['ctr']
y = D['y']; Et = D['Et']
print(f'clean rows in train pool: {int(is_clean.sum())} | containment label median (clean): {np.median(CONT[is_clean]):.3f}')

clean rows in train pool: 30303 | containment label median (clean): 0.946


In [3]:
CFG = dict(d=128, nhead=4, layers=3, dropout=0.1, lr=3e-4, wd=1e-4, batch=96)
LAM_C = 0.5
class SubNetCA(nn.Module):
    def __init__(self, in_dim, la0, lb0):
        super().__init__()
        d = CFG['d']
        self.embed = nn.Linear(in_dim, d)
        layer = nn.TransformerEncoderLayer(d, CFG['nhead'], dim_feedforward=4*d,
                                           dropout=CFG['dropout'], batch_first=True)
        self.enc = nn.TransformerEncoder(layer, CFG['layers'], enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + NG, d), nn.ReLU(), nn.Dropout(CFG['dropout']), nn.Linear(d, 3))
        self.fhead = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
        self.chead = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
        self.la = nn.Parameter(torch.tensor(float(la0))); self.lb = nn.Parameter(torch.tensor(float(lb0)))
    def forward(self, x, m, g, ecell):
        h = self.enc(self.embed(x), src_key_padding_mask=~m)
        w = torch.sigmoid(self.fhead(h).squeeze(-1)) * m.float()
        base = self.la * torch.log1p((w * ecell).sum(1, keepdim=True)) + self.lb
        wm = m.unsqueeze(-1).float()
        p = self.norm((h * wm).sum(1) / wm.sum(1).clamp(min=1))
        return base + self.head(torch.cat([p, g], 1)), self.chead(p).squeeze(-1)
QS = torch.tensor([0.25, 0.5, 0.75], device=DEVICE)
def pinball(q, yb):
    d = yb - q
    return torch.maximum(QS * d, (QS - 1) * d).mean()
def wcalib(qv, qt, yva):
    wv = qv[:, 2] - qv[:, 0]; wt_ = qt[:, 2] - qt[:, 0]
    cuts = np.quantile(wv, [1/3, 2/3])
    gv = np.digitize(wv, cuts); gt = np.digitize(wt_, cuts)
    pe = np.empty(len(qt))
    for g in range(3):
        if (gv == g).sum() < 10 or (gt == g).sum() == 0:
            a, b2 = np.polyfit(qv[:, 1], yva, 1)
        else:
            a, b2 = np.polyfit(qv[gv == g, 1], yva[gv == g], 1)
        pe[gt == g] = np.exp(a * qt[gt == g, 1] + b2)
    return pe
def train_eval(seed, epochs, patience):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = SubNetCA(D['IN_DIM'], D['la0'], D['lb0']).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tr_idx = np.concatenate([np.asarray(ktr), ctr])
    ck = CKPT / f'nb51_caux_s{seed}.pt'
    def batches(idx, bs, sh):
        idx = np.asarray(idx)
        if sh: idx = rng.permutation(idx)
        for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
    def fwd(b): return model(T['X'][b], T['M'][b], T['G'][b], T['E'][b])
    def loss_fn(q, ch, b):
        L = pinball(q, T['Y'][b])
        ic = T['IC'][b]
        if ic.sum() > 0:
            L = L + LAM_C * (((ch - T['C'][b]) ** 2) * ic).sum() / ic.sum()
        return L
    def run(idx):
        model.eval(); qs = []; cs = []
        with torch.no_grad():
            for b in batches(idx, 256, False):
                q, ch = fwd(b); qs.append(q.cpu().numpy()); cs.append(ch.cpu().numpy())
        return np.concatenate(qs), np.concatenate(cs)
    def vloss():
        model.eval(); s = 0.0; k = 0
        with torch.no_grad():
            for b in batches(kva, 256, False):
                q, ch = fwd(b)
                s += pinball(q, T['Y'][b]).item(); k += 1
        return s / max(k, 1)
    best = 1e9; bstate = None; wait = 0; ep0 = 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        model.load_state_dict(st['model']); opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best = st['best']; bstate = st['bstate']; wait = st['wait']; ep0 = st['ep'] + 1
        rng = np.random.default_rng(seed + 1000 * ep0)
        print(f'  resume s{seed} from epoch {ep0}', flush=True)
    for ep in range(ep0, epochs):
        model.train()
        for b in batches(tr_idx, CFG['batch'], True):
            opt.zero_grad()
            q, ch = fwd(b)
            loss_fn(q, ch, b).backward()
            opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(model.state_dict()); wait = 0
        else: wait += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        best=best, bstate=bstate, wait=wait, ep=ep), ck)
        if wait >= patience: break
    model.load_state_dict(bstate)
    qv, cv = run(kva); qt, ct_ = run(kte)
    qc, cc = run(ctr)
    ccorr = float(np.corrcoef(cc, CONT[ctr])[0, 1])
    pe = wcalib(qv, qt, y[kva])
    return float(resolution(pe, Et[kte])['sigma_eff']), pe, ccorr

In [4]:
EPOCHS = {'smoke': 2, 'full': 100}[MODE]
PATIENCE = {'smoke': 99, 'full': 15}[MODE]
SEEDS = {'smoke': [0], 'full': [0, 1]}[MODE]
TAG = '' if MODE == 'full' else '_smoke'
CSVP = OUT / f'nb51_caux{TAG}.csv'
done = set()
if CSVP.exists():
    done = set(pd.read_csv(CSVP)['seed'])
    print('resume, done:', sorted(done))
for seed in SEEDS:
    if seed in done: print('skip', seed); continue
    t1 = time.time()
    sig, pe, ccorr = train_eval(seed, EPOCHS, PATIENCE)
    np.save(OUT / f'nb51_pred{TAG}_caux_s{seed}.npy', pe)
    row = dict(seed=seed, sigma_eff=round(sig, 4), ccorr=round(ccorr, 3), elapsed=round(time.time()-t1))
    pd.DataFrame([row]).to_csv(CSVP, mode='a', header=not CSVP.exists() or CSVP.stat().st_size == 0, index=False)
    print(f'caux seed {seed}: sigma_eff {sig:.4f} | corr(chat, c) clean {ccorr:.3f} ({row["elapsed"]}s)', flush=True)
print(pd.read_csv(CSVP).to_string(index=False))

caux seed 0: sigma_eff 0.0444 | corr(chat, c) clean 0.658 (1328s)


caux seed 1: sigma_eff 0.0448 | corr(chat, c) clean 0.609 (1265s)


 seed  sigma_eff  ccorr  elapsed
    0     0.0444  0.658     1328
    1     0.0448  0.609     1265


## Verdict vs nb44

Anchors: nb44 singles 0.0442 +/- 0.0005, stack record 0.0425 (Act 3a: 0.0419). Win = beat singles mean by >0.002 overall or in any E>17 bin; the aux head must also actually learn containment (corr(chat, c) >> 0) for any gain to be attributed to the mechanism.

In [5]:
te_e = Et[kte]
edges = np.quantile(te_e, np.linspace(0, 1, 7))
def perbin(pe):
    out = []
    for i in range(6):
        hi = edges[i+1] + (1e-9 if i == 5 else 0)
        mm = (te_e >= edges[i]) & (te_e < hi)
        out.append(resolution(pe[mm], te_e[mm])['sigma_eff'])
    return out
print('nb44 singles 0.0442 +/- 0.0005 | stack 0.0425 | per-bin (TTA) 0.0657/0.0483/0.0364/0.0361/0.0345/0.0354')
preds = [np.load(OUT / f'nb51_pred{TAG}_caux_s{s}.npy') for s in SEEDS
         if (OUT / f'nb51_pred{TAG}_caux_s{s}.npy').exists()]
if preds:
    sig = [resolution(p, te_e)['sigma_eff'] for p in preds]
    ens = np.stack(preds).mean(0)
    print(f'caux mean {np.mean(sig):.4f} +/- {np.std(sig):.4f} | ens {resolution(ens, te_e)["sigma_eff"]:.4f}')
    print('per-bin ' + ' / '.join(f'{b:.4f}' for b in perbin(ens)))

nb44 singles 0.0442 +/- 0.0005 | stack 0.0425 | per-bin (TTA) 0.0657/0.0483/0.0364/0.0361/0.0345/0.0354
caux mean 0.0446 +/- 0.0002 | ens 0.0435


per-bin 0.0678 / 0.0481 / 0.0369 / 0.0364 / 0.0354 / 0.0380
